# Indiana Bosonic Code - DMANH

In [1]:
from jaqalpaq import run
from jaqalpaq import emulator
from jaqalpaq.run import run_jaqal_file, run_jaqal_string, run_jaqal_batch, run_jaqal_circuit, frontend

from jaqalpaq.emulator.unitary import UnitarySerializedEmulator
emulator_backend = UnitarySerializedEmulator()

#from Experiment import Experiment
import numpy as np
import csv
from datetime import datetime, date, time, timezone
from pytz import timezone
from collections import OrderedDict
import pytz
from scipy.special import jn_zeros
import math
import matplotlib.pyplot as plt
import pickle as pkl
from jaqalpaq.parser import parse_jaqal_string


def timestamp_generate():
    mountain = timezone('US/Mountain')
    timestamp_utc = datetime.utcnow()
    timestamp_local = timestamp_utc.astimezone(mountain)
    return(timestamp_utc, timestamp_local)

def int_to_base(n, b):
    if n == 0:
        return "0"
    digits = []
    is_negative = n < 0
    n = abs(n)
    while n:
        digits.append(int(n % b))
        n //= b
    if is_negative:
        digits.append("-")
    return ''.join(str(x) if x < 10 else chr(55 + x) for x in reversed(digits))



In [ ]:
#Can import a file here with the angles in it, so angles can be called automatically. 

repeats = 1000
override_dict={ "__repeats__":repeats,}
#beta_list = [0]

import os
import ast

working_dir = os.getcwd()
recursive_dir = working_dir
for ii in range(5):
    temp_dir = os.path.dirname(recursive_dir)
    recursive_dir = temp_dir
file_path = recursive_dir


#with open(os.path.join(file_path,'Indiana_QSD_Dict1.txt'), 'r') as file:
#    string_data = file.read()
#angles = ast.literal_eval(string_data)

# ok, making this part up now as it doesn't exist, so how 
# exactly this is called in should be modified to match the actual file
angles = np.load('../build/notebook/angles.npy')  # shape (3, 49)
# betas should be: 0 -0.4 0 -0.2 0 0 0 0.2 0 0.4
betas = np.load('../build/notebook/betas.npy')  # shape (N, 2)
beta_list = [complex(b[0], b[1]) for b in betas]

#timestep = 3 #length of the code
timesteps = angles.shape[1]

# Added beta parameter - JBG
def generate_jaqal_header(beta):
    sandia_beta = 0.5j * beta
    header = '''from Calibration_PulseDefinitions.QubitBosonPulses usepulses *

let reBeta {sandia_beta.real}
let imBeta {sandia_beta.imag}
let imMeas 1

register q[2]

prepare_all
'''
    return header

def generate_jaqal_code(timesteps, beta):
    circuit = ''
    for max in range(timesteps): 
        circuit += '// Timestep %s \n' %(max)
        for time in range(max+1):
            circuit += '// Circuit %s \n' %(time)
            circuit += 'xCD q[0] 1 1 %s %s \n' %(angles[0, time], angles[1, time])
            circuit += 'Rz q[0] %s \n' % (angles[2, time])
            circuit += 'xCD q[0] 1 1 %s %s \n' %(-angles[0, time], -angles[1, time])
            circuit += 'xCD q[0] 1 1 %s %s \n' %(-angles[0, time], -angles[1, time])
            circuit += 'Rz q[0] %s \n' % (angles[2, time])
            circuit += 'xCD q[0] 1 1 %s %s \n' %(angles[0, time], angles[1, time])

    return circuit

def generate_jaqal_detect():
    
    detect = '''// Characteristic-function readout.
loop imMeas {
R q[1] 0 1.5707963267948966
}
xCD q[1] 1 1 reBeta imBeta
measure_all'''
    return detect

In [3]:
#Make the jaqal files looping over the betas such that each one is a different "batch" 

circuit_list=[] # list of circuits for the subset, divided into batches
num_batch=len(beta_list)
for beta in beta_list:
    header = generate_jaqal_header()
    circuits = generate_jaqal_code(3, beta) 
    footer = generate_jaqal_detect()
    circuit_list.append(header + circuits+footer)



In [7]:
#This will not run on the emulator, so leaving that if statement out. 


print(circuit_list)
result_list=[]
#for i in range(len(beta_list)+1):
for jaqal_circ in circuit_list: #looping over batches
    result_list.append(run.run_jaqal_string(jaqal_circ, overrides = override_dict))
    
    #print(jaqal_circ)


['from Calibration_PulseDefinitions.QubitBosonPulses usepulses *\n\nlet reBeta -0.20000000000000001\nlet imBeta 0\nlet imMeas 1\n\nregister q[2]\n\nprepare_all\n// Timestep 0 \n// Circuit 0 \nxCD q[0] 1 1 -0.18512 0 \nRz q[0] -0.7999999999999998 \nxCD q[0] 1 1 0.18512 0 \nxCD q[0] 1 1 0.18512 0 \nRz q[0] -0.7999999999999998 \nxCD q[0] 1 1 -0.18512 0 \n// Timestep 1 \n// Circuit 0 \nxCD q[0] 1 1 -0.18512 0 \nRz q[0] -0.7999999999999998 \nxCD q[0] 1 1 0.18512 0 \nxCD q[0] 1 1 0.18512 0 \nRz q[0] -0.7999999999999998 \nxCD q[0] 1 1 -0.18512 0 \n// Circuit 1 \nxCD q[0] 1 1 -0.1812894914003226 -0.03746377861097782 \nRz q[0] -0.7999999999999998 \nxCD q[0] 1 1 0.1812894914003226 0.03746377861097782 \nxCD q[0] 1 1 0.1812894914003226 0.03746377861097782 \nRz q[0] -0.7999999999999998 \nxCD q[0] 1 1 -0.1812894914003226 -0.03746377861097782 \n// Timestep 2 \n// Circuit 0 \nxCD q[0] 1 1 -0.18512 0 \nRz q[0] -0.7999999999999998 \nxCD q[0] 1 1 0.18512 0 \nxCD q[0] 1 1 0.18512 0 \nRz q[0] -0.7999999999

In [ ]:
#Extract the data from the results object
#This extracts the axis data

chi = True
measure_im_chi = True
ajc = False

reMeas = []
imMeas = []
if chi:
    for ii, _ in enumerate(re_values):
        reMeas.append(results[0].by_subbatch[ii].by_subcircuit[0].probability_by_int[:,0])
        if measure_im_chi:
            imMeas.append(results[1].by_subbatch[ii].by_subcircuit[0].probability_by_int[:,0])

bsb_results = []
if ajc: 
    for ii, _ in enumerate(angles):
        bsb_results.append(BSB_results[0].by_subbatch[ii].by_subcircuit[0].probability_by_int[:,0])


if chi:
    #Save all probabilities
    np.save('prob_reMeas',reMeas)
    np.save('prob_imMeas',imMeas)

if chi:
    #Save beta ranges
    np.save('ReBetas',re_values)
    np.save('ImBetas',im_values)

if ajc:
    #Save blue sideband data
    np.save('bsb_probs',bsb_results)
    np.save('bsb_angles',angles)


In [ ]:
#This gets the characteristic function data
if chi:

    def exp_z(prob): #Postselect the results on the probe qubit dependent on prep qubit being in spin down
        #probe qubit's state probabilities
        if prob[0] + prob[2] == 0:
            state0 = 0
            state1 = 0
        else:
            state0 = prob[0]/(prob[0]+prob[2])
            state1 = prob[2]/(prob[0]+prob[2])
        
        return state0 - state1
    
    
    def exp_z_spin_up(prob): #Postselect the results on the probe qubit dependent on prep qubit being in spin up
        #probe qubit's state probabilities
        if prob[1] + prob[3] == 0:
            state0 = 0
            state1 = 0
        else:
            state0 = prob[1]/(prob[1]+prob[3])
            state1 = prob[3]/(prob[1]+prob[3])
        
        return state0 - state1
    
    expZ_reMeas = []
    expZ_imMeas = []
    expZ_reMeas_prep_spin_up = []
    expZ_imMeas_prep_spin_up = []
    
    for ii, _ in enumerate(re_values):
        expZ_reMeas.append(exp_z(reMeas[ii]))
        expZ_reMeas_prep_spin_up.append(exp_z_spin_up(reMeas[ii]))
        if measure_im_chi:
            expZ_imMeas.append(exp_z(imMeas[ii])) 
            expZ_imMeas_prep_spin_up.append(exp_z_spin_up(imMeas[ii])) 

    #Save <Z>
    np.save('expZ_reMeas',expZ_reMeas)
    np.save('expZ_imMeas',expZ_imMeas) 


In [ ]:
#Plotting

if chi:   
    #Plot the real and imaginary parts of the characteristic function for the prep qubit being spin DOWN
    #Note the example plot is from a squeezing operation (not a displacement)
    x=re_values
    y=im_values
    xsp = 2*(re_beta_list[1]-re_beta_list[0])
    ysp = 2*(im_beta_list[1]-im_beta_list[0])
    print(xsp,ysp)
    verts = [[-xsp,-ysp],[xsp,-ysp],[xsp,ysp],[-xsp,ysp]]
    
    print(expZ_reMeas[0])
    
    iter = 0
    Xindices = [r for r in range(len(re_beta_list))]
    Yindices = [r for r in range(len(im_beta_list))]
    
    Zre = np.empty((len(Xindices),len(Yindices)))
    Zim = np.empty((len(Xindices),len(Yindices)))
    
    for i in Xindices:
        for j in Yindices:
            Zre[i,j] = expZ_reMeas[iter]
            if measure_im_chi:
                Zim[i,j] = expZ_imMeas[iter]
            iter +=1
    
    fig, ax = plt.subplots(figsize = (6.5,5))
    zmax = abs(Zre.max())
    h=ax.pcolormesh(re_beta_list, im_beta_list, Zre.T, cmap = 'bwr', rasterized = True, vmin = -zmax, vmax = zmax)
    ax.set_xlabel('Re[$\\beta$]')
    ax.set_ylabel('Im[$\\beta$]')
    cbar = fig.colorbar(h)
    cbar.ax.set_ylim((-zmax,zmax))
    fig.savefig("Re_chi_beta.png", dpi=300, bbox_inches="tight")
    plt.show()
    
    fig, ax = plt.subplots(figsize = (6.5,5))
    h = plt.pcolormesh(re_beta_list, im_beta_list, Zim.T, cmap = 'bwr', rasterized = True, vmin = -zmax, vmax = zmax)
    plt.xlabel('Re[$\\beta$]')
    plt.ylabel('Im[$\\beta$]')
    cbar = fig.colorbar(h)
    cbar.ax.set_ylim((-zmax,zmax))
    fig.savefig("Im_chi_beta.png", dpi=300, bbox_inches="tight")
    plt.show()

